### Voice Pipeline — Stage by Stage

Building a production-ready VoicePipeline class.

Layers:
1. STT — audio → text
2. RAG — text → answer  
3. TTS — answer → audio

Learning notebook. Production .py comes later.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

from IPython.display import Audio, display

In [2]:
load_dotenv()

# OpenAI client — used by listen() and speak()
client = OpenAI()

# Create llm 
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Embeddings for the Chroma store
embeddings = OpenAIEmbeddings()

# Load the domain knowledge base
vectorstore = Chroma(
    persist_directory=r'C:\Users\USER\rag_course\chroma_db_domain',
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are AgriVoice, a farming assistant for Nigerian farmers. '
     'Answer using ONLY the context below. Keep it under 3 sentences. '
     'If the answer is not in the context, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])

rag_chain = rag_prompt | llm | StrOutputParser()
print(f'RAG ready - {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


RAG ready - 82 chunks


In [3]:
class VoicePipeline:
    '''
    One voice turn: audio in -> STT -> RAG -> TTS -> audio out.
    '''
    
    # Store config once so every method can reuse it without reloading
    def __init__(self, language: str='English'):
        '''Setup the pipeline once'''
    
        self.language = language
        print(f'VoicePipeline created (language={self.language})')
        
    # Convert a recorded audio file into text (STT)
    def listen(self, audio_path: str) -> str:
        with open(audio_path, 'rb') as f:
            transcript = client.audio.transcriptions.create(
                model='whisper-1',
                file=f
            )
        return transcript.text
    
    # Send the farmer's question through the multilingual RAG chain
    def think(self, question: str) -> str:
        
        # 1. Question → English
        question_en = translate_to_english(question, self.language)
        
        # 2. RAG in English
        docs = retriever.invoke(question)
        context = '\n\n'.join(doc.page_content for doc in docs)
        answer_en = rag_chain.invoke({
            'context': context,
            'question': question
        })
        
        # 3. Answer → user's language
        answer_local = translate(answer_en, self.language)
        return answer_local
    
    # Convert the answer text into spoken audio bytes (TTS)
    def speak(self, answer: str) -> bytes:
        response = client.audio.speech.create(
            model='tts-1',
            voice='alloy',
            input=answer
        )
        return response.content
    
    # Chain listen → think → speak — the full one-turn voice flow
    def turn(self, audio_path: str) -> bytes:
        print('🎧 Listening...')
        question = self.listen(audio_path)
        print(f'   📝 {question}')
        
        print('🧠 Thinking...')
        answer = self.think(question)
        print(f'   💬 {answer}')
        
        print('🔊 Speaking...')
        audio_bytes = self.speak(answer)
        
        return audio_bytes
    

vp = VoicePipeline(language='Yoruba')
print(f'Pipeline language: {vp.language}')

VoicePipeline created (language=Yoruba)
Pipeline language: Yoruba


In [ ]:
vp = VoicePipeline(language='English')

audio = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'

# Listen
transcript = vp.listen(audio_path=audio)
print(f'📝 Transcript: {transcript}')

VoicePipeline created (language=English)
📝 Transcript: How do I treat cassava mosaic disease?


In [ ]:
# Think
answer = vp.think('How do I treat cassava mosaic disease?')
print(f'💬 Answer: {answer}')
print()

answer = vp.think('What is the capital of France?')
print(f'💬 {answer}')

💬 Answer: Use disease-free cuttings for planting, plant resistant cassava varieties, and remove and destroy infected plants immediately. Control whitefly vectors with appropriate insecticides and avoid planting cassava close to infected fields.

💬 I don't know.


In [ ]:
vp = VoicePipeline(language='English')

audio_bytes = vp.speak('Use disease-free cuttings and remove infected plants early.')

print(f'🔊 Got {len(audio_bytes)} bytes of audio')

# Save it so we can play it
out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\speak_test.mp3'

with open(out_path, 'wb') as f:
    f.write(audio_bytes)
print(f'💾 Saved to: {out_path}')

VoicePipeline created (language=English)
🔊 Got 60672 bytes of audio
💾 Saved to: C:\Users\USER\rag_course\12b_voice_agents\data\speak_test.mp3


In [ ]:
vp = VoicePipeline(language='English')

audio_in = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'
audio_out = r'C:\Users\USER\rag_course\12b_voice_agents\data\turn_test.mp3'

audio_bytes = vp.turn(audio_in)

with open(audio_out, 'wb') as f:
    f.write(audio_bytes)

print(f'💾 Saved: {audio_out}')

VoicePipeline created (language=English)
🎧 Listening...
   📝 How do I treat cassava mosaic disease?
🧠 Thinking...
   💬 Use disease-free cuttings for planting, plant resistant cassava varieties, and remove and destroy infected plants immediately. Control whitefly vectors with appropriate insecticides and avoid planting cassava close to infected fields.
🔊 Speaking...
💾 Saved: C:\Users\USER\rag_course\12b_voice_agents\data\turn_test.mp3


In [ ]:
display(Audio(audio_out))

In [ ]:
# Terminology store for correct crop name translation
from langchain_core.documents import Document

TERMINOLOGY_DIR = r'C:\Users\USER\rag_course\chroma_db_terminology'

terminology_store = Chroma(
    persist_directory=TERMINOLOGY_DIR,
    embedding_function=embeddings,
    collection_name='terminology',
)
terminology_retriever = terminology_store.as_retriever(search_kwargs={'k': 5})

print(f'✅ Terminology store ready — {terminology_store._collection.count()} docs')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Terminology store ready — 36 docs


In [ ]:
def get_relevant_terms(text: str) -> str:
    docs = terminology_retriever.invoke(text)
    return '\n'.join(f'- {doc.page_content}' for doc in docs)

def translate(text: str, target_lang: str) -> str:
    if target_lang == 'English':
        return text
    
    terms = get_relevant_terms(text)
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following English text to {target_lang}.\n\n'
        f'MANDATORY VOCABULARY (use these exact words):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY {target_lang} — no English mixed in, except proper nouns.\n'
        f'2. Use the correct tone marks and diacritics.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
          {'role': 'system', 'content': system},
          {'role': 'user', 'content': text},  
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()


def translate_to_english(text: str, source_lang: str) -> str:
    if source_lang == 'English':
        return text
    
    terms = get_relevant_terms(text)
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following {source_lang} text to English.\n\n'
        f'MANDATORY VOCABULARY (use these to decode crops/diseases):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY English.\n'
        f'2. Use the terminology above to correctly identify crops.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print('✅ translate() and translate_to_english() ready')

✅ translate() and translate_to_english() ready


In [6]:
needed = [
    'client', 'llm', 'embeddings',
    'vectorstore', 'retriever', 'rag_chain',
    'terminology_store', 'terminology_retriever',
    'get_relevant_terms', 'translate', 'translate_to_english',
    'VoicePipeline',
]

for name in needed:
    status = '✅' if name in globals() else '❌ MISSING'
    print(f'{status}  {name}')

✅  client
✅  llm
✅  embeddings
✅  vectorstore
✅  retriever
✅  rag_chain
❌ MISSING  terminology_store
❌ MISSING  terminology_retriever
❌ MISSING  get_relevant_terms
❌ MISSING  translate
❌ MISSING  translate_to_english
✅  VoicePipeline


In [8]:
from langchain_core.documents import Document

# Terminology Chroma store for correct crop/disease translation
TERMINOLOGY_DIR = r'C:\Users\USER\rag_course\chroma_db_terminology'

terminology_store = Chroma(
    persist_directory=TERMINOLOGY_DIR,
    embedding_function=embeddings,
    collection_name='terminology',
)
terminology_retriever = terminology_store.as_retriever(search_kwargs={'k': 5})

print(f'✅ Terminology store ready — {terminology_store._collection.count()} docs')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Terminology store ready — 36 docs


In [9]:
def get_relevant_terms(text: str) -> str:
    
    docs = terminology_retriever.invoke(text)
    return '\n'.join(f'- {doc.page_content}' for doc in docs)

def translate(text: str, target_lang: str) -> str:
    
    if target_lang == 'English':
        return text
    
    terms = get_relevant_terms(text)
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following English text to {target_lang}.\n\n'
        f'MANDATORY VOCABULARY (use these exact words):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY {target_lang} — no English mixed in, except proper nouns.\n'
        f'2. Use the correct tone marks and diacritics.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

# Translate FROM a Nigerian language TO English
def translate_to_english(text: str, source_lang: str) -> str:
        
    if source_lang == 'English':
        return text
    
    terms = get_relevant_terms(text)
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following {source_lang} text to English.\n\n'
        f'MANDATORY VOCABULARY (use these to decode crops/diseases):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY English.\n'
        f'2. Use the terminology above to correctly identify crops.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print('✅ Translation helpers ready')


✅ Translation helpers ready


In [10]:
needed = [
    'client', 'llm', 'embeddings',
    'vectorstore', 'retriever', 'rag_chain',
    'terminology_store', 'terminology_retriever',
    'get_relevant_terms', 'translate', 'translate_to_english',
    'VoicePipeline',
]

for name in needed:
    status = '✅' if name in globals() else '❌ MISSING'
    print(f'{status}  {name}')

✅  client
✅  llm
✅  embeddings
✅  vectorstore
✅  retriever
✅  rag_chain
✅  terminology_store
✅  terminology_retriever
✅  get_relevant_terms
✅  translate
✅  translate_to_english
✅  VoicePipeline


In [11]:
vp = VoicePipeline(language='English')
audio_in = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'
audio_bytes = vp.turn(audio_in)

out = r'C:\Users\USER\rag_course\12b_voice_agents\data\turn_en.mp3'
with open(out, 'wb') as f:
    f.write(audio_bytes)
print(f'💾 {out}')

VoicePipeline created (language=English)
🎧 Listening...
   📝 How do I treat cassava mosaic disease?
🧠 Thinking...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


   💬 To treat cassava mosaic disease, use disease-free cuttings for planting, select resistant cassava varieties, and immediately remove and destroy infected plants. Control whitefly vectors with appropriate insecticides and avoid planting cassava near infected fields.
🔊 Speaking...
💾 C:\Users\USER\rag_course\12b_voice_agents\data\turn_en.mp3


In [12]:
vp = VoicePipeline(language='Yoruba')
print(vp.think('How do I treat cassava mosaic disease?'))
print()

vp = VoicePipeline(language='Hausa')
print(vp.think('How do I treat cassava mosaic disease?'))

VoicePipeline created (language=Yoruba)


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Lati tọju àrùn mosaic ẹgẹ́, lo awọn gige ti ko ni àrùn fun ikore, yan awọn irugbin ẹgẹ́ ti o ni resistance, ki o si yọkuro lẹsẹkẹsẹ ki o si pa awọn ọgbin ti o ni àrùn. Ṣakoso kòkòrò funfun pẹlu awọn insecticides to yẹ ki o si yago fun ikore ẹgẹ́ nitosi awọn ilẹ̀ ti o ni àrùn.

VoicePipeline created (language=Hausa)
Yi amfani da yankan da ba su da cuta don shuka, shuka nau'in rogo masu juriya, kuma cire da lalata shukar da ta kamu da cuta nan take. Sarrafa farar kwari tare da magungunan kashe kwari masu dacewa kuma ka guji shuka rogo kusa da filayen da suka kamu da cuta.
